# CalCOFI Big Data Analytics Solution
## One-year-ahead station-level hypoxia warning

This Google Colab notebook implements the A5 BDAS workflow with **PySpark and Spark MLlib**. It predicts whether a regularly sampled CalCOFI station will have a low minimum dissolved-oxygen observation in the following survey year.

**Project scope:** California Current observations, years 2000 onward. The analysis uses station-year aggregates so that the prediction unit is a station in one predictor year and the outcome belongs to the same station in the following year.

Run cells from top to bottom. Download this notebook and the generated `output` folder when finished; these form the Colab-code/data-output submission evidence.

In [ ]:
# Colab includes a PySpark runtime. Do not install a different version because it can conflict with Colab's Dataproc packages.
import pyspark
print('Using PySpark version:', pyspark.__version__)


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator, RegressionEvaluator
from pyspark.ml.feature import Imputer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.ml.regression import LinearRegression, RandomForestRegressor

spark = (SparkSession.builder
         .appName('CalCOFI_BDAS')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
spark.sparkContext.setLogLevel('WARN')

OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', context='notebook')
print('Spark version:', spark.version)


## 1. Collect initial data

The two CalCOFI input archives are versioned with this notebook in the GitHub repository under `A5/datasets/`. This cell downloads them to the temporary Colab runtime and extracts the original CSV files. No manual upload is required.

In [ ]:
# Download the repository versions once per Colab runtime, then extract them for Spark.
from urllib.request import urlretrieve

REPOSITORY_DATA_URL = 'https://raw.githubusercontent.com/graham-gc/infosys722/main/A5/datasets'
# Change to True when the archives in GitHub have been updated during this Colab runtime.
REFRESH_DATA = False
DATA_DIR = Path('/content/datasets')
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVES = ['194903-202105_Bottle.csv.zip', '194903-202105_Cast.csv.zip']

for archive_name in ARCHIVES:
    archive_path = DATA_DIR / archive_name
    if REFRESH_DATA or not archive_path.exists():
        urlretrieve(f'{REPOSITORY_DATA_URL}/{archive_name}', archive_path)
    with zipfile.ZipFile(archive_path, 'r') as zf:
        zf.extractall(DATA_DIR)

SEARCH_DIRS = [DATA_DIR]

def locate_file(candidates):
    for directory in SEARCH_DIRS:
        for name in candidates:
            path = directory / name
            if path.exists():
                return str(path)
    raise FileNotFoundError(f'Could not find any of: {candidates}. Upload the Bottle and Cast CSV files, then rerun this cell.')

BOTTLE_PATH = locate_file(['194903-202105_Bottle.csv'])
CAST_PATH = locate_file(['194903-202105_Cast.csv'])
print('Bottle:', BOTTLE_PATH)
print('Cast:', CAST_PATH)


In [ ]:
def read_calcofi_csv(path):
    return (spark.read
            .option('header', True)
            .option('inferSchema', True)
            .option('mode', 'PERMISSIVE')
            .csv(path))

bottle_raw = read_calcofi_csv(BOTTLE_PATH)
cast_raw = read_calcofi_csv(CAST_PATH)

collection_summary = pd.DataFrame([
    {'table': 'Bottle', 'source_file': Path(BOTTLE_PATH).name, 'rows': bottle_raw.count(), 'columns': len(bottle_raw.columns)},
    {'table': 'Cast', 'source_file': Path(CAST_PATH).name, 'rows': cast_raw.count(), 'columns': len(cast_raw.columns)},
])
display(collection_summary)
collection_summary.to_csv(OUTPUT_DIR / '01_collection_summary.csv', index=False)

print('Bottle schema')
bottle_raw.printSchema()
print('Cast schema')
cast_raw.printSchema()


## 2. Describe, explore and verify data quality

The project uses direct dissolved oxygen (`O2ml_L`) as the authoritative environmental measurement. Temperature, salinity and nutrients are predictor candidates. The quality report below makes missingness explicit before any rows are removed.

In [ ]:
bottle_fields = ['Cst_Cnt', 'Sta_ID', 'Depthm', 'T_degC', 'Salnty', 'O2ml_L', 'O2Sat', 'PO4uM', 'NO3uM', 'SiO3uM', 'ChlorA', 'RecInd']
cast_fields = ['Cst_Cnt', 'Year', 'Month', 'Lat_Dec', 'Lon_Dec', 'Bottom_D', 'Sta_Code', 'Data_Type']

missing_rows = []
for field in bottle_fields:
    if field in bottle_raw.columns:
        total = bottle_raw.count()
        missing = bottle_raw.filter(F.col(field).isNull()).count()
        missing_rows.append({'field': field, 'missing_rows': missing, 'completeness_pct': round(100 * (1 - missing / total), 3)})
quality_report = pd.DataFrame(missing_rows).sort_values('completeness_pct')
display(quality_report)
quality_report.to_csv(OUTPUT_DIR / '02_bottle_completeness.csv', index=False)

recind_distribution = (bottle_raw.groupBy('RecInd').count().orderBy('RecInd').toPandas()) if 'RecInd' in bottle_raw.columns else pd.DataFrame()
display(recind_distribution)
recind_distribution.to_csv(OUTPUT_DIR / '03_record_indicator_distribution.csv', index=False)


In [ ]:
# Raw-data visualisations: year coverage and dissolved-oxygen distribution before preparation.
cast_years = cast_raw.select('Cst_Cnt', 'Year').dropDuplicates(['Cst_Cnt'])
raw_oxygen = bottle_raw.select('Cst_Cnt', F.col('O2ml_L').cast('double').alias('O2ml_L')).join(cast_years, 'Cst_Cnt', 'inner').filter(F.col('Year') >= 2000)
raw_sample = raw_oxygen.filter(F.col('O2ml_L').isNotNull()).sample(False, 0.02, seed=722).limit(100000).toPandas()
year_counts = raw_oxygen.groupBy('Year').count().orderBy('Year').toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.lineplot(data=year_counts, x='Year', y='count', marker='o', ax=axes[0], color='#1f5a99')
axes[0].set_title('Raw oxygen observations by survey year')
axes[0].set_ylabel('Bottle observations')
sns.histplot(raw_sample['O2ml_L'], bins=60, ax=axes[1], color='#1f5a99')
axes[1].axvline(1.4, color='crimson', linestyle='--', label='Project warning threshold')
axes[1].set_title('Raw dissolved oxygen distribution, year ≥ 2000')
axes[1].set_xlabel('O2ml_L')
axes[1].legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '04_raw_data_exploration.png', dpi=220, bbox_inches='tight')
plt.show()


## 3. Data preparation: select, clean, integrate, construct and reformat

The Cast table provides time and location; the Bottle table provides sampled water chemistry. They are integrated using `Cst_Cnt`. The primary analysis keeps observed bottle records (`RecInd = 3` where that field is present), years 2000 onward, plausible oxygen values (0–12 mL/L), and rows with a station identifier and coordinates. The aggregation is intentional vertical reduction: many bottle depths become one comparable station-year record.

The one-year-ahead outcome is the **minimum observed oxygen in the same station in the following calendar year**. `1.4 mL/L` is a project warning threshold, not a universal ecological mortality threshold.

In [ ]:
# Select required fields and make names unambiguous before the join.
bottle = bottle_raw.select(*[c for c in bottle_fields if c in bottle_raw.columns])
cast = cast_raw.select(*[c for c in cast_fields if c in cast_raw.columns])

# Cast_Cnt should identify one sampling event. Deduplicating protects the one-to-many Cast-to-Bottle merge.
cast = cast.dropDuplicates(['Cst_Cnt'])

prepared = (bottle
    .join(cast, on='Cst_Cnt', how='inner')
    .withColumn('Year', F.col('Year').cast('int'))
    .withColumn('O2ml_L', F.col('O2ml_L').cast('double'))
    .withColumn('T_degC', F.col('T_degC').cast('double'))
    .withColumn('Salnty', F.col('Salnty').cast('double'))
    .withColumn('Depthm', F.col('Depthm').cast('double'))
    .withColumn('PO4uM', F.col('PO4uM').cast('double'))
    .withColumn('NO3uM', F.col('NO3uM').cast('double'))
    .withColumn('SiO3uM', F.col('SiO3uM').cast('double'))
    .filter(F.col('Year') >= 2000)
    .filter(F.col('Sta_ID').isNotNull() & F.col('Lat_Dec').isNotNull() & F.col('Lon_Dec').isNotNull())
    .filter(F.col('O2ml_L').between(0, 12)))

if 'RecInd' in prepared.columns:
    prepared = prepared.filter(F.col('RecInd').cast('int') == 3)

print('Prepared bottle records:', prepared.count())
print('Prepared Cast events:', prepared.select('Cst_Cnt').distinct().count())


In [ ]:
# Construct station-year predictors. Mean and minimum oxygen represent different aspects of current conditions.
station_year = (prepared
    .groupBy('Sta_ID', 'Year')
    .agg(
        F.avg('Lat_Dec').alias('Lat_Dec'),
        F.avg('Lon_Dec').alias('Lon_Dec'),
        F.avg('Depthm').alias('Mean_Depthm'),
        F.avg('T_degC').alias('T_degC_Mean'),
        F.avg('Salnty').alias('Salnty_Mean'),
        F.avg('O2ml_L').alias('Current_O2_Mean'),
        F.min('O2ml_L').alias('Current_O2_Min'),
        F.avg('PO4uM').alias('PO4_Mean'),
        F.avg('NO3uM').alias('NO3_Mean'),
        F.avg('SiO3uM').alias('SiO3_Mean'),
        F.count('*').alias('Observation_Count'),
        F.countDistinct('Cst_Cnt').alias('Cast_Count')
    ))

# Self-join only the following year at the same station. This prevents same-year leakage.
current = station_year.alias('current')
following = station_year.select('Sta_ID', F.col('Year').alias('Target_Year'), F.col('Current_O2_Min').alias('Target_O2_Min')).alias('following')
model_data = (current.join(
    following,
    (F.col('current.Sta_ID') == F.col('following.Sta_ID')) & (F.col('current.Year') + 1 == F.col('following.Target_Year')),
    'inner')
    .select('current.*', 'following.Target_Year', 'following.Target_O2_Min')
    .withColumn('Hypoxia_Next_Year', F.when(F.col('Target_O2_Min') < 1.4, F.lit(1.0)).otherwise(F.lit(0.0)))
    .withColumn('Log_PO4', F.log1p(F.greatest(F.col('PO4_Mean'), F.lit(0.0))))
    .withColumn('Log_NO3', F.log1p(F.greatest(F.col('NO3_Mean'), F.lit(0.0))))
    .withColumn('Log_SiO3', F.log1p(F.greatest(F.col('SiO3_Mean'), F.lit(0.0))))
    .withColumn('Sqrt_Depth', F.sqrt(F.greatest(F.col('Mean_Depthm'), F.lit(0.0)))))

model_summary = model_data.agg(
    F.count('*').alias('station_year_pairs'),
    F.countDistinct('Sta_ID').alias('stations'),
    F.min('Year').alias('first_predictor_year'),
    F.max('Year').alias('last_predictor_year'),
    F.mean('Hypoxia_Next_Year').alias('next_year_warning_rate')
).toPandas()
display(model_summary)
model_summary.to_csv(OUTPUT_DIR / '05_station_year_model_summary.csv', index=False)


## 4. Reduction and projection

The core feature set is deliberately small: location, depth, temperature, salinity, preceding-year oxygen, and record counts. Nutrient measures are transformed with `log1p` because they are right-skewed. Depth is square-root transformed for the same reason. The extended model later tests whether the nutrient features add predictive value.

In [ ]:
CORE_FEATURES = [
    'Lat_Dec', 'Lon_Dec', 'Sqrt_Depth', 'T_degC_Mean', 'Salnty_Mean',
    'Current_O2_Mean', 'Current_O2_Min', 'Observation_Count', 'Cast_Count'
]
EXTENDED_FEATURES = CORE_FEATURES + ['Log_PO4', 'Log_NO3', 'Log_SiO3']

# The time split emulates deployment: all predictor years before the last two observed years train the model.
available_years = [r['Year'] for r in model_data.select('Year').distinct().orderBy('Year').collect()]
if len(available_years) < 4:
    raise ValueError('At least four predictor years are required for a train/validation/test split.')
train_end_year = available_years[-3]
validation_year = available_years[-2]
test_year = available_years[-1]

train_df = model_data.filter(F.col('Year') <= train_end_year).cache()
validation_df = model_data.filter(F.col('Year') == validation_year).cache()
test_df = model_data.filter(F.col('Year') == test_year).cache()

split_summary = pd.DataFrame([
    {'split': 'train', 'predictor_years': f'{available_years[0]}–{train_end_year}', 'station_year_pairs': train_df.count()},
    {'split': 'validation', 'predictor_years': str(validation_year), 'station_year_pairs': validation_df.count()},
    {'split': 'test', 'predictor_years': str(test_year), 'station_year_pairs': test_df.count()},
])
display(split_summary)
split_summary.to_csv(OUTPUT_DIR / '06_temporal_split.csv', index=False)


## 5–7. Modelling, test design and data mining

The regression task predicts next-year minimum oxygen. The classification task produces a warning where next-year minimum oxygen is below 1.4 mL/L. Linear/logistic regression provide transparent baselines; random forests test nonlinear relationships. The final test year remains unseen until model selection is complete.

In [ ]:
def make_pipeline(feature_columns, estimator, label_col, prediction_col='prediction'):
    imputed = [f'{c}__imputed' for c in feature_columns]
    imputer = Imputer(inputCols=feature_columns, outputCols=imputed, strategy='median')
    assembler = VectorAssembler(inputCols=imputed, outputCol='features', handleInvalid='skip')
    estimator = estimator.setLabelCol(label_col).setFeaturesCol('features').setPredictionCol(prediction_col)
    return Pipeline(stages=[imputer, assembler, estimator])

def regression_metrics(predictions):
    return {
        'RMSE': RegressionEvaluator(labelCol='Target_O2_Min', predictionCol='prediction', metricName='rmse').evaluate(predictions),
        'MAE': RegressionEvaluator(labelCol='Target_O2_Min', predictionCol='prediction', metricName='mae').evaluate(predictions),
        'R2': RegressionEvaluator(labelCol='Target_O2_Min', predictionCol='prediction', metricName='r2').evaluate(predictions),
    }

def classification_metrics(predictions):
    return {
        'ROC_AUC': BinaryClassificationEvaluator(labelCol='Hypoxia_Next_Year', rawPredictionCol='rawPrediction', metricName='areaUnderROC').evaluate(predictions),
        'F1': MulticlassClassificationEvaluator(labelCol='Hypoxia_Next_Year', predictionCol='prediction', metricName='f1').evaluate(predictions),
        'Accuracy': MulticlassClassificationEvaluator(labelCol='Hypoxia_Next_Year', predictionCol='prediction', metricName='accuracy').evaluate(predictions),
    }

# Exploratory models are evaluated on the validation year only.
regression_candidates = {
    'Linear regression': LinearRegression(maxIter=100, regParam=0.05, elasticNetParam=0.1),
    'Random forest regression': RandomForestRegressor(numTrees=200, maxDepth=8, seed=722),
}
classification_candidates = {
    'Logistic regression': LogisticRegression(maxIter=100, regParam=0.05),
    'Random forest classifier': RandomForestClassifier(numTrees=250, maxDepth=8, seed=722),
}

exploration_rows = []
for name, estimator in regression_candidates.items():
    fitted = make_pipeline(CORE_FEATURES, estimator, 'Target_O2_Min').fit(train_df)
    metrics = regression_metrics(fitted.transform(validation_df))
    exploration_rows.append({'task': 'regression', 'model': name, **metrics})

for name, estimator in classification_candidates.items():
    fitted = make_pipeline(CORE_FEATURES, estimator, 'Hypoxia_Next_Year').fit(train_df)
    metrics = classification_metrics(fitted.transform(validation_df))
    exploration_rows.append({'task': 'classification', 'model': name, **metrics})

exploration_results = pd.DataFrame(exploration_rows)
display(exploration_results)
exploration_results.to_csv(OUTPUT_DIR / '07_validation_model_comparison.csv', index=False)


In [ ]:
# Select the random forests after exploratory comparison, retrain using train + validation data, then evaluate once on the held-out final year.
train_validation_df = train_df.unionByName(validation_df).cache()
final_regression = make_pipeline(
    CORE_FEATURES, RandomForestRegressor(numTrees=400, maxDepth=8, minInstancesPerNode=3, seed=722), 'Target_O2_Min'
).fit(train_validation_df)
final_classifier = make_pipeline(
    CORE_FEATURES, RandomForestClassifier(numTrees=400, maxDepth=8, minInstancesPerNode=3, seed=722), 'Hypoxia_Next_Year'
).fit(train_validation_df)

test_regression = final_regression.transform(test_df)
test_classification = final_classifier.transform(test_df)

test_metrics = pd.DataFrame([
    {'task': 'regression', 'model': 'Random forest', **regression_metrics(test_regression)},
    {'task': 'classification', 'model': 'Random forest', **classification_metrics(test_classification)},
])
display(test_metrics)
test_metrics.to_csv(OUTPUT_DIR / '08_final_test_metrics.csv', index=False)


## 8. Evaluate patterns, visualise results and iterate

The map ranks held-out stations by the model's warning probability. Squares mark model warnings at probability ≥ 0.50; circles mark the observed following-year warning outcome. Overlap is a correct warning. The feature-importance chart is descriptive rather than causal.

In [ ]:
# Extract prediction probability and create a station-level output table for the report.
prediction_output = (test_classification
    .select('Sta_ID', 'Year', 'Target_Year', 'Lat_Dec', 'Lon_Dec', 'Current_O2_Mean', 'Current_O2_Min', 'Target_O2_Min',
            'Hypoxia_Next_Year', 'probability', 'prediction')
    .withColumn('Warning_Probability', vector_to_array('probability')[1])
    .withColumn('Model_Warning', F.when(F.col('Warning_Probability') >= 0.50, 1).otherwise(0)))
prediction_pd = prediction_output.toPandas()
prediction_pd.to_csv(OUTPUT_DIR / '09_held_out_station_predictions.csv', index=False)
display(prediction_pd.sort_values('Warning_Probability', ascending=False).head(20))

fig, ax = plt.subplots(figsize=(10, 8))
scatter = ax.scatter(prediction_pd['Lon_Dec'], prediction_pd['Lat_Dec'], c=prediction_pd['Warning_Probability'],
                     cmap='YlOrRd', s=55, alpha=0.9, label='All held-out stations')
warned = prediction_pd[prediction_pd['Model_Warning'] == 1]
observed = prediction_pd[prediction_pd['Hypoxia_Next_Year'] == 1]
ax.scatter(warned['Lon_Dec'], warned['Lat_Dec'], facecolors='none', edgecolors='purple', marker='s', s=180, linewidths=1.8, label='Model warning')
ax.scatter(observed['Lon_Dec'], observed['Lat_Dec'], facecolors='none', edgecolors='black', s=180, linewidths=1.8, label='Observed next-year warning')
plt.colorbar(scatter, ax=ax, label='Predicted warning probability')
ax.set(title=f'Retrospective one-year-ahead warnings: predictor year {test_year}', xlabel='Longitude', ylabel='Latitude')
ax.legend(loc='best')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '10_held_out_warning_map.png', dpi=240, bbox_inches='tight')
plt.show()


In [ ]:
# Permutation-style Spark feature importance from the selected random-forest models.
reg_rf = final_regression.stages[-1]
clf_rf = final_classifier.stages[-1]
importance = pd.DataFrame({
    'feature': CORE_FEATURES,
    'regression_importance': reg_rf.featureImportances.toArray(),
    'classification_importance': clf_rf.featureImportances.toArray(),
}).sort_values('classification_importance', ascending=False)
importance.to_csv(OUTPUT_DIR / '11_feature_importance.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=importance.sort_values('regression_importance'), x='regression_importance', y='feature', ax=axes[0], color='#4c78a8')
axes[0].set(title='Regression feature importance', xlabel='Importance', ylabel='')
sns.barplot(data=importance.sort_values('classification_importance'), x='classification_importance', y='feature', ax=axes[1], color='#e15759')
axes[1].set(title='Classification feature importance', xlabel='Importance', ylabel='')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / '12_feature_importance.png', dpi=240, bbox_inches='tight')
plt.show()


In [ ]:
# Iteration: test whether adding transformed nutrient predictors improves final-year performance.
extended_regression = make_pipeline(
    EXTENDED_FEATURES, RandomForestRegressor(numTrees=400, maxDepth=8, minInstancesPerNode=3, seed=722), 'Target_O2_Min'
).fit(train_validation_df)
extended_classifier = make_pipeline(
    EXTENDED_FEATURES, RandomForestClassifier(numTrees=400, maxDepth=8, minInstancesPerNode=3, seed=722), 'Hypoxia_Next_Year'
).fit(train_validation_df)

iteration_results = pd.DataFrame([
    {'feature_set': 'Core', 'task': 'regression', **regression_metrics(test_regression)},
    {'feature_set': 'Extended nutrients', 'task': 'regression', **regression_metrics(extended_regression.transform(test_df))},
    {'feature_set': 'Core', 'task': 'classification', **classification_metrics(test_classification)},
    {'feature_set': 'Extended nutrients', 'task': 'classification', **classification_metrics(extended_classifier.transform(test_df))},
])
display(iteration_results)
iteration_results.to_csv(OUTPUT_DIR / '13_iteration_core_vs_extended.csv', index=False)


## 9. Export submission evidence

The final cell packages the reproducible evidence: CSV tables and PNG figures produced by the notebook. Download the notebook itself from Colab (`File → Download → Download .ipynb`) and include it in the same submission zip. Do not claim a live operational warning: these are retrospective model outputs and need local threshold governance, expert review and future prospective validation.

In [ ]:
# Package outputs. Add the downloaded .ipynb to this archive before Canvas submission.
archive_path = shutil.make_archive('/content/A5_CalCOFI_BDAS_outputs', 'zip', OUTPUT_DIR)
print('Created:', archive_path)
from google.colab import files
files.download(archive_path)
